In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
# from DG_classes import *
from diffusion_geometry.operators import (
    derivative_weak,
    hessian_02_sym_weak,
    hessian_02_weak,
    hessian_coords,
    hessian_functions,
    hessian_operator,
    levi_civita_02_weak,
    levi_civita_11_weak,
    lie_bracket_weak,
    up_delta_weak,
)
import matplotlib.pyplot as plt
import plotly.figure_factory as ff
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from diffusion_geometry.visualisation import *
import imageio
from scipy.sparse import diags, bmat

from methods.pde import *

# import warnings
# # Suppress specific warnings
# warnings.filterwarnings("ignore", category=DeprecationWarning)

# Vector Calculus: 2D Plane vs 2D Manifold

Let's compare vector calculus on a 2D plane in R² versus on a 2D manifold like a torus.

In [3]:
## 1. Vector Calculus on 2D Plane in R²

# Define an interesting polynomial function p(x,y)
def p(x, y):
    """Polynomial function: p(x,y) = x³ - 3xy² + 2x²y - y³ + x² + y²"""
    return x**3 - 3*x*y**2 + 2*x**2*y - y**3 + x**2 + y**2

# Define the gradient of p
def grad_p(x, y):
    """Gradient of p: ∇p = (∂p/∂x, ∂p/∂y)"""
    dpdy = 6*x**2*y - 3*y**2 + 2*x**2 - 3*y**2 + 2*y
    dpdx = 3*x**2 - 3*y**2 + 4*x*y + 2*x
    return dpdx, dpdy

# Create grid for visualization
x_range = np.linspace(-2, 2, 100)  # shape: (100,)
y_range = np.linspace(-2, 2, 100)  # shape: (100,)
X, Y = np.meshgrid(x_range, y_range)  # shape: (100, 100) each

# Evaluate function on grid
Z = p(X, Y)  # shape: (100, 100)

# Plot 1: Function p(x,y) as a continuous heatmap
fig1 = go.Figure(data=go.Heatmap(
    x=x_range,
    y=y_range,
    z=Z,
    colorscale=["#166dde", "lightgray", "#e32636"],
    showscale=False
))

fig1.update_layout(
    title='',
    width=800,
    height=600,
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(
        showgrid=False,
        showline=False,
        zeroline=False,
        showticklabels=False,
        title='',
        range=[-2, 2]
    ),
    yaxis=dict(
        showgrid=False,
        showline=False,
        zeroline=False,
        showticklabels=False,
        title='',
        range=[-2, 2]
    ),
    # shapes=[
    #     # X-axis line
    #     dict(
    #         type="line",
    #         x0=-2, y0=0, x1=2, y1=0,
    #         line=dict(color="black", width=3),
    #         layer="above"
    #     ),
    #     # Y-axis line
    #     dict(
    #         type="line",
    #         x0=0, y0=-2, x1=0, y1=2,
    #         line=dict(color="black", width=3),
    #         layer="above"
    #     )
    # ]
)
fig1.update_yaxes(scaleanchor="x", scaleratio=1)
fig1.show()

# Create coarser grid for vector field visualization
x_vec = np.linspace(-2, 2, 20)  # shape: (20,)
y_vec = np.linspace(-2, 2, 20)  # shape: (20,)
X_vec, Y_vec = np.meshgrid(x_vec, y_vec)  # shape: (20, 20) each

# Evaluate gradient on vector grid
dpdx, dpdy = grad_p(X_vec, Y_vec)  # shape: (20, 20) each

# Plot 2: Gradient vector field
fig2 = ff.create_quiver(
    X_vec, Y_vec, dpdx, dpdy,
    scale=0.02,
    arrow_scale=0.3,
    name='∇p',
    line=dict(color='black', width=2)
)

fig2.update_layout(
    title='',
    width=800,
    height=600,
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(
        showgrid=False,
        showline=False,
        zeroline=False,
        showticklabels=False,
        title='',
        range=[-2, 2]
    ),
    yaxis=dict(
        showgrid=False,
        showline=False,
        zeroline=False,
        showticklabels=False,
        title='',
        range=[-2, 2]
    )
)
fig2.update_yaxes(scaleanchor="x", scaleratio=1)
fig2.show()

In [4]:
## 2. Vector Calculus on a 2D Manifold (Torus)

# Torus parameterization: (u, v) -> (x, y, z)
def torus_parametrization(u, v, R=2, r=1):
    """
    Torus parameterization with major radius R and minor radius r
    u, v are parameters in [0, 2π]
    Returns (x, y, z) coordinates
    """
    x = (R + r * np.cos(v)) * np.cos(u)  # shape: same as u, v
    y = (R + r * np.cos(v)) * np.sin(u)  # shape: same as u, v
    z = r * np.sin(v)  # shape: same as u, v
    return x, y, z

# Define a function on the torus in parameter space (u, v)
def f_torus(u, v):
    """Function on torus: f(u,v) = sin(2u) + cos(3v) + 0.5*sin(u)*cos(v)"""
    return np.sin(2*u) + np.cos(3*v) + 0.5*np.sin(u)*np.cos(v)

# Create parameter grid for torus
u_range = np.linspace(0, 2*np.pi, 60)  # shape: (60,)
v_range = np.linspace(0, 2*np.pi, 40)  # shape: (40,)
U, V = np.meshgrid(u_range, v_range)  # shape: (40, 60) each

# Get torus coordinates
X_torus, Y_torus, Z_torus = torus_parametrization(U, V)  # shape: (40, 60) each

# Evaluate function on torus
F_torus = f_torus(U, V)  # shape: (40, 60)

# Create 3D torus mesh with function values as colors
fig_torus = go.Figure(data=go.Surface(
    x=X_torus,
    y=Y_torus,
    z=Z_torus,
    surfacecolor=F_torus,
    colorscale=["#166dde", "lightgray", "#e32636"],
    showscale=False,
    lighting=dict(
        ambient=1,
        diffuse=1,
        specular=1
    ),
    lightposition=dict(x=0, y=0, z=1)
))

fig_torus.update_layout(
    title='',
    width=800,
    height=600,
    scene=dict(
        xaxis=dict(showgrid=False, showticklabels=False, title='', showbackground=False, range=[-3, 3]),
        yaxis=dict(showgrid=False, showticklabels=False, title='', showbackground=False, range=[-3, 3]),
        zaxis=dict(showgrid=False, showticklabels=False, title='', showbackground=False, range=[-3, 3]),
        bgcolor='white',
        aspectmode='cube'
    ),
    paper_bgcolor='white',
    plot_bgcolor='white'
)

fig_torus.show()

In [5]:
# Compute gradient of f_torus in parameter space
def grad_f_torus(u, v):
    """Gradient of f in parameter space (u, v)"""
    dfdu = 2*np.cos(2*u) + 0.5*np.cos(u)*np.cos(v)  # shape: same as u, v
    dfdv = -3*np.sin(3*v) - 0.5*np.sin(u)*np.sin(v)  # shape: same as u, v
    return dfdu, dfdv

# Compute tangent vectors to torus surface
def torus_tangent_vectors(u, v, R=2, r=1):
    """Compute tangent vectors ∂r/∂u and ∂r/∂v for torus surface"""
    # ∂r/∂u
    dxdu = -(R + r * np.cos(v)) * np.sin(u)  # shape: same as u, v
    dydu = (R + r * np.cos(v)) * np.cos(u)   # shape: same as u, v
    dzdu = np.zeros_like(u)                  # shape: same as u, v
    
    # ∂r/∂v
    dxdv = -r * np.sin(v) * np.cos(u)  # shape: same as u, v
    dydv = -r * np.sin(v) * np.sin(u)  # shape: same as u, v
    dzdv = r * np.cos(v)               # shape: same as u, v
    
    return (dxdu, dydu, dzdu), (dxdv, dydv, dzdv)

# Create coarser grid for vector field
u_vec = np.linspace(0, 2*np.pi, 30)  # shape: (30,)
v_vec = np.linspace(0, 2*np.pi, 30)  # shape: (30,)
U_vec, V_vec = np.meshgrid(u_vec, v_vec)  # shape: (30, 30) each

# Get torus coordinates for vector field
X_vec_torus, Y_vec_torus, Z_vec_torus = torus_parametrization(U_vec, V_vec)  # shape: (30, 30) each

# Compute gradient in parameter space
dfdu_vec, dfdv_vec = grad_f_torus(U_vec, V_vec)  # shape: (30, 30) each

# Get tangent vectors
(dxdu, dydu, dzdu), (dxdv, dydv, dzdv) = torus_tangent_vectors(U_vec, V_vec)  # shape: (30, 30) each

# Project gradient onto tangent space (gradient vector on surface)
grad_x = dfdu_vec * dxdu + dfdv_vec * dxdv  # shape: (30, 30)
grad_y = dfdu_vec * dydu + dfdv_vec * dydv  # shape: (30, 30)
grad_z = dfdu_vec * dzdu + dfdv_vec * dzdv  # shape: (30, 30)

# Create 3D quiver plot
fig_grad_torus = go.Figure()

# Add torus surface (full opacity, white)
fig_grad_torus.add_trace(go.Surface(
    x=X_torus,
    y=Y_torus,
    z=Z_torus,
    colorscale=[[0, 'white'], [1, 'white']],
    showscale=False,
    opacity=1.0,
    lighting=dict(ambient=0.8, diffuse=0.5, specular=0.1)
))

# Add gradient vectors as 3D arrows
scale_factor = 0.1
for i in range(U_vec.shape[0]):
    for j in range(U_vec.shape[1]):
        # Start point
        x0, y0, z0 = X_vec_torus[i,j], Y_vec_torus[i,j], Z_vec_torus[i,j]
        # End point
        x1 = x0 + scale_factor * grad_x[i,j]
        y1 = y0 + scale_factor * grad_y[i,j]
        z1 = z0 + scale_factor * grad_z[i,j]
        
        # Add arrow as scatter3d line
        fig_grad_torus.add_trace(go.Scatter3d(
            x=[x0, x1],
            y=[y0, y1],
            z=[z0, z1],
            mode='lines+markers',
            line=dict(color='black', width=4),
            marker=dict(size=[2, 6], color=['black', 'black']),
            showlegend=False
        ))

fig_grad_torus.update_layout(
    title='',
    width=800,
    height=600,
    scene=dict(
        xaxis=dict(showgrid=False, showticklabels=False, title='', showbackground=False, range=[-3, 3]),
        yaxis=dict(showgrid=False, showticklabels=False, title='', showbackground=False, range=[-3, 3]),
        zaxis=dict(showgrid=False, showticklabels=False, title='', showbackground=False, range=[-3, 3]),
        bgcolor='white',
        aspectmode='cube'
    ),
    paper_bgcolor='white',
    plot_bgcolor='white'
)

fig_grad_torus.show()